<a href="https://colab.research.google.com/github/Davidson-Open-University/csc203-labs/blob/main/notebooks/CSC203-Lab7-Counting-by-Computer.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CSC 203 Lab 7: Counting by Computer

**Study Session 7: Applications to Counting** · Davidson Open University Global

**In this lab you will:**
- apply the product and sum rules, and check them by listing cases;
- count permutations and combinations with Python’s math module;
- count arrangements with repeated letters;
- use inclusion–exclusion and the pigeonhole principle, and build rows of Pascal’s triangle.

**Time:** about 60 minutes in four parts:
- **Part A – Explore** (about 10 minutes): run the examples and read the explanations.
- **Part B – Practise** (about 20 minutes): fill in the blanks. A self-check tells you if you are right.
- **Part C – Build a tool** (about 20 minutes): complete a short Python function and test it.
- **Part D – Apply and reflect** (about 10 minutes): use your tools on your own example and write short answers.

**How to use this notebook**
1. Run each code cell in order: click it and press **Shift + Enter** (or the ▶ button).
2. Where you see `...`, replace it with your own code, run the cell, then run the **check** cell under it.
   - ✅ means correct. ❌ gives you a hint: try again. ⏳ means you have not filled in the blank yet.
   - The checks never show the answer, so you learn by working it out. You can try as many times as you like.
3. Stuck? Read the hint, look back at the study session, or ask in the discussion forum. Do not skip ahead: each part builds on the last.
4. Save your work: in Colab choose **File → Save a copy in GitHub** and save it to your own `csc203-labs` repository. Offline in Jupyter, press **Ctrl + S** and later upload the file with **Add file → Upload files** on GitHub.

*This lab is practice to help you learn and prepare for the assignment. It is not graded unless your tutor tells you otherwise.*

In [ ]:
#@title ▶ Run this cell first: it sets up the self-checks (you do not need to read it)
# You do not need to read or change this code.
import hashlib as _h
class _Blank(Exception): pass
def _norm(x, unordered=False):
    try:
        import numpy as _np
        if isinstance(x, _np.ndarray): x = x.tolist()
        elif isinstance(x, _np.bool_): x = bool(x)
        elif isinstance(x, _np.integer): x = int(x)
        elif isinstance(x, _np.floating): x = float(x)
    except ImportError:
        pass
    if x is Ellipsis or x is None: raise _Blank()
    if isinstance(x, bool): return 'b:' + str(x)
    if isinstance(x, (int, float)): return 'n:' + repr(round(float(x), 6) + 0.0)
    if isinstance(x, str): return 't:' + ' '.join(x.lower().split())
    if isinstance(x, (set, frozenset)): return 's{' + ','.join(sorted(_norm(i) for i in x)) + '}'
    if isinstance(x, (list, tuple)):
        items = [_norm(i) for i in x]
        if unordered: items.sort()
        return 'l[' + ','.join(items) + ']'
    if isinstance(x, dict): return 'd{' + ','.join(sorted(_norm(k) + '=' + _norm(v) for k, v in x.items())) + '}'
    try:
        import sympy as _sp
        if isinstance(x, _sp.Basic):
            x = _sp.sympify(x)
            if x.is_number: return 'n:' + repr(round(float(x), 6) + 0.0)
            syms = sorted(x.free_symbols, key=str)
            vals = []
            for t in (0.3, 0.7, 1.3):
                v = complex(x.subs({s: t + 0.11 * i for i, s in enumerate(syms)}).evalf())
                vals.append(round(v.real, 6) + 0.0)
            return 'e:' + repr(vals)
    except ImportError:
        pass
    return 'r:' + repr(x)
def _hash(q, s): return _h.sha256(('csc203|' + q + '|' + s).encode()).hexdigest()[:16]
def check(q, value):
    """Check your answer to question q. Prints ✅, ❌ with a hint, or ⏳ if the blank is not filled yet."""
    spec = _CHECKS[q]
    try:
        if callable(value) and 'args' in spec:
            outs = []
            for a in spec['args']:
                try:
                    outs.append(value(*a))
                except _Blank:
                    raise
                except Exception as e:
                    print(f'❌ Question {q}: your function stopped with an error: {type(e).__name__}: {e}')
                    return
            if all(o is None or o is Ellipsis for o in outs): raise _Blank()
            got = _norm(outs, False)
        elif spec.get('logic'):
            if value is None or value is Ellipsis: raise _Blank()
            import sympy as _sp
            from itertools import product as _p
            syms = sorted(_sp.sympify(value).free_symbols, key=str)
            got = 'L' + ''.join('1' if bool(_sp.sympify(value).subs(dict(zip(syms, v)))) else '0' for v in _p([True, False], repeat=len(syms))) + ':' + ','.join(map(str, syms))
        else:
            got = _norm(value, spec.get('unordered', False))
    except _Blank:
        print(f'⏳ Question {q}: fill in the blank (replace ...) and run the cell above, then run this cell again.')
        return
    if _RECORD is not None:
        _RECORD[q] = _hash(q, got); print(f'✅ Question {q}: recorded.'); return
    if _hash(q, got) == spec['h']:
        print(f'✅ Question {q}: correct! ' + spec.get('ok', 'Well done.'))
    else:
        print(f'❌ Question {q}: not yet. Hint: ' + spec['hint'])
_RECORD = None

_CHECKS = {
    '7.1': {'h': '7d073ec5b8f07dcd', 'hint': 'Multiply the number of choices for each of the 8 positions: 26 × 26 × 26 × 10 × 10 × 10 × 26 × 26.', 'ok': 'That is nearly 12 billion plates.'},
    '7.2': {'h': 'fdd949410b63de4b', 'hint': 'sum(1 for n in range(100, 1000) if len(set(str(n))) == 3)', 'ok': 'Check with the product rule: 9 choices for the first digit (not 0), then 9, then 8.'},
    '7.3': {'h': '6eb5e7b46543d382', 'hint': 'set(...) removes the repeated arrangements. Check: 5!/2! for ABUJA.', 'ok': 'With a letter repeated k times, divide by k!.'},
    '7.4': {'h': '8913130be08d8f2f', 'hint': 'The numbers divisible by both 3 and 5 are the multiples of 15.', 'ok': 'Brute force: sum(1 for n in range(1, 1001) if n % 3 == 0 or n % 5 == 0) gives the same answer.'},
    '7.5': {'h': 'ad42f3fb4ce347b6', 'hint': 'Share 400 students as evenly as possible among 12 months. Some month must get at least the average, rounded up.', 'ok': 'However the birthdays fall, at least this many students share a month.'},
    '7.6': {'h': '279617ec15bab41d', 'hint': 'The next row is [1] + [row[i] + row[i + 1] for i in range(len(row) - 1)] + [1].', 'ok': 'Check: the numbers in row n add up to 2ⁿ.', 'args': [(1,), (4,), (6,), (10,)]},
    '7.7': {'h': 'f688ce6f7ad8b0c7', 'hint': "Count the women in a committee c with sum(p.startswith('W') for p in c).", 'ok': 'Brute force and the formula agree.'},
}
print('Self-checks ready. Now carry on with Part A.')

## Part A – Explore (about 10 minutes)
Counting formulas can be checked by **brute force**: list every case with `itertools` and count them. Run the cell to check the product rule and the formulas for permutations and combinations.

- `product(...)`: every way of choosing one item from each list (the **product rule**);
- `permutations(items, r)`: ordered selections of r items, P(n, r) = n!/(n − r)!;
- `combinations(items, r)`: unordered selections of r items, C(n, r) = n!/(r!(n − r)!).

In [ ]:
from itertools import product, permutations, combinations
from math import comb, perm, factorial

meals = ['rice', 'yam', 'beans']
drinks = ['water', 'zobo', 'kunu', 'juice']
print('Meal-and-drink pairs:', len(list(product(meals, drinks))), '= 3 × 4')

letters = 'ABCDE'
print('Ordered choices of 3 from 5 :', len(list(permutations(letters, 3))), '  perm(5, 3) =', perm(5, 3))
print('Unordered choices of 3 from 5:', len(list(combinations(letters, 3))), '  comb(5, 3) =', comb(5, 3))

**Order matters** for permutations (ABC and CBA are different), but not for combinations (ABC and CBA are the same choice). That is why P(5, 3) = 3! × C(5, 3).

## Part B – Practise (about 20 minutes)
**B1. Number plates.** A Nigerian number plate has the pattern **3 letters, 3 digits, 2 letters**, for example ABC-123-XY. Use the product rule to find how many plates are possible (26 letters, 10 digits, repetition allowed).

In [ ]:
answer_7_1 = ...

In [ ]:
check('7.1', answer_7_1)

**B2. Distinct digits.** How many 3-digit numbers (100 to 999) have three **different** digits? Count them by brute force with a comprehension over `range(100, 1000)`. *Hint:* a number n has distinct digits when `len(set(str(n))) == 3`.

In [ ]:
answer_7_2 = ...

In [ ]:
check('7.2', answer_7_2)

**B3. Repeated letters.** The letters of LAGOS are all different, so they have 5! arrangements. ABUJA has two A's, so swapping them gives nothing new. Count the **distinct** arrangements of each word with `len(set(permutations(word)))`, and give the answer as a tuple `(LAGOS, ABUJA)`.

In [ ]:
answer_7_3 = ...

In [ ]:
check('7.3', answer_7_3)

**B4. Inclusion–exclusion.** How many numbers from 1 to 1000 are divisible by 3 **or** 5? Use |A ∪ B| = |A| + |B| − |A ∩ B|, where A = multiples of 3 and B = multiples of 5. (`1000 // 3` gives the number of multiples of 3.) Then check by brute force.

In [ ]:
answer_7_4 = ...

In [ ]:
check('7.4', answer_7_4)

**B5. Pigeonhole principle.** A school has 400 students. What is the **smallest** number of students who must share the same birth month? *Hint:* ⌈400/12⌉, which Python writes as `-(-400 // 12)` or `math.ceil(400 / 12)`.

In [ ]:
answer_7_5 = ...

In [ ]:
check('7.5', answer_7_5)

## Part C – Build a tool (about 20 minutes)
### C1. Pascal's triangle
Row n of Pascal's triangle is C(n, 0), C(n, 1), …, C(n, n). Each inside entry is the sum of the two entries above it (**Pascal's identity**: C(n, k) = C(n − 1, k − 1) + C(n − 1, k)).

Complete `pascal_row(n)` **without** using `comb`: start from `row = [1]`, and repeat n times: make the next row from the current one by adding neighbouring pairs, with a 1 at each end.

In [ ]:
def pascal_row(n):
    row = [1]
    for _ in range(n):
        row = ...    # YOUR CODE: the next row, made from the current row
    return row

for n in range(6):
    print(pascal_row(n))

In [ ]:
check('7.6', pascal_row)

### C2. Choosing a committee with conditions
A class has 6 women and 5 men. A committee of 4 must include **at least 2 women**. Count the committees by brute force: label the people, use `combinations`, and keep the committees with at least 2 women. Then check your answer with the formula C(6,2)C(5,2) + C(6,3)C(5,1) + C(6,4).

In [ ]:
people = ['W1', 'W2', 'W3', 'W4', 'W5', 'W6', 'M1', 'M2', 'M3', 'M4', 'M5']
answer_7_7 = ...   # YOUR CODE: count the 4-person committees with at least 2 women

In [ ]:
check('7.7', answer_7_7)

## Part D – Apply and reflect (about 10 minutes)
**D1. Prepare for Assignment 7.** Use `comb`, `perm` and the product rule to check your answers to the committee and code questions in the assignment. For the committee questions, also check one answer by brute force with `combinations` or `permutations`.

In [ ]:
# D1: check your Assignment 7 answers

**D2. Reflect.** Answer in one or two sentences each:
1. Explain, with an example, when order matters and when it does not.
2. In B4, why do we subtract the multiples of 15?
3. Brute force worked well here. Why would it fail for counting the number plates in B1?

*Type your answer here (double-click this cell to edit it).*

## Finished? Save your work
In Colab: **File → Save a copy in GitHub** → your `csc203-labs` repository → commit message *"Lab 7 completed"*.
Offline: save with **Ctrl + S**, then upload the file to your repository with **Add file → Upload files** when you are online.

Then go back to Moodle and mark the lab as done.